In [ ]:
!pip install datasets requests torch

In [ ]:
import time
import random
import statistics
import requests
import torch

from datasets import load_dataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

reporting = load_dataset("JayShah07/reporting_final_dataset")

print("Dataset structure:")
print(reporting)

print("\nDataset sizes:")
print(f"Train: {len(reporting['train'])}")
print(f"Validation: {len(reporting['validation'])}")
print(f"Test: {len(reporting['test'])}")

In [ ]:
TEST_SAMPLES = 130

test_queries = [
    example["query"]
    for example in reporting["test"].select(range(TEST_SAMPLES))
]

print(f"Loaded {len(test_queries)} test queries")
print("\nSample queries:")
for q in test_queries[:5]:
    print("-", q)


In [ ]:
API_URL = "https://classification-model-mlops.onrender.com/api/predict"

HEADERS = {
    "accept": "application/json",
    "Content-Type": "application/json"
}


In [ ]:
latencies = []
success = 0
failures = 0

print("🚀 Starting online inference using TEST set...\n")

for idx, query in enumerate(test_queries, start=1):
    payload = {"text": query}
    start_time = time.time()

    try:
        response = requests.post(
            API_URL,
            json=payload,
            headers=HEADERS,
            timeout=10
        )

        latency = time.time() - start_time
        latencies.append(latency)

        if response.status_code == 200:
            success += 1
            result = response.json()

            print(
                f"[{idx:03}] ✅ "
                f"Latency={latency:.3f}s | "
                f"Module={result['module_best']} | "
                f"Date={result['date_best']}"
            )
        else:
            failures += 1
            print(f"[{idx:03}] ❌ HTTP {response.status_code}")

    except Exception as e:
        failures += 1
        print(f"[{idx:03}] ❌ Exception: {e}")

    # Simulate real user traffic
    time.sleep(random.uniform(0.2, 0.6))


In [ ]:
print("\n================ SUMMARY ================")
print(f"Total requests   : {len(test_queries)}")
print(f"Successful calls : {success}")
print(f"Failures         : {failures}")

if latencies:
    print(f"Average latency  : {statistics.mean(latencies):.3f}s")
    print(f"P95 latency      : {statistics.quantiles(latencies, n=20)[18]:.3f}s")
    print(f"Max latency      : {max(latencies):.3f}s")
